# Modelling: predicting the age of an abalone

We compare a `LinearRegression` baseline with a `RandomForestRegressor` on the number of `Rings`, and track both runs with MLflow. The steps are written as functions so they can be moved to modules later.

In [1]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

2026/10/23 03:24:58 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\Dell7670\workspace\xhec-mlops-2026-project\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


In [2]:
def load_data(path):
    return pd.read_csv(path)


def preprocess(df, encoder=None):
    """Rename columns to snake_case and one-hot encode `sex`. Fits an encoder if none is given."""
    df = df.rename(columns=lambda c: c.lower().replace(" ", "_"))
    if encoder is None:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        encoder.fit(df[["sex"]])
    sex = pd.DataFrame(
        encoder.transform(df[["sex"]]),
        columns=encoder.get_feature_names_out(["sex"]),
        index=df.index,
    )
    features = pd.concat([df.drop(columns=["sex", "rings"]), sex], axis=1)
    return features, df["rings"], encoder


def train_model(model, x_train, y_train):
    return model.fit(x_train, y_train)


def evaluate(model, x_test, y_test):
    pred = model.predict(x_test)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
        "mae": float(mean_absolute_error(y_test, pred)),
        "r2": float(r2_score(y_test, pred)),
    }

In [3]:
df = load_data("../data/abalone.csv")
x, y, encoder = preprocess(df)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(x_train.shape, x_test.shape)
x_train.head()

(3341, 10) (836, 10)


,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,sex_F,sex_I,sex_M
4038,0.550,0.445,0.125,0.6720,0.2880,0.1365,0.210,0.0,1.0,0.0
1272,0.475,0.355,0.100,0.5035,0.2535,0.0910,0.140,0.0,1.0,0.0
3384,0.305,0.225,0.070,0.1485,0.0585,0.0335,0.045,1.0,0.0,0.0
3160,0.275,0.200,0.065,0.1165,0.0565,0.0130,0.035,0.0,1.0,0.0
3894,0.495,0.380,0.135,0.6295,0.2630,0.1425,0.215,0.0,0.0,1.0


## MLflow tracking

Runs are stored locally in `mlflow.db` at the repo root (git-ignored). Open the UI from the repo root with:

```
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db
```

In [4]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("abalone-age")

models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(n_estimators=100, random_state=42),
}

# Our own models: trust the tree type so MLflow can serialize the random forest.
trusted = ["sklearn.tree._tree.Tree"]
results = {}
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        mlflow.log_params(model.get_params())
        train_model(model, x_train, y_train)
        results[name] = evaluate(model, x_test, y_test)
        mlflow.log_metrics(results[name])
        mlflow.sklearn.log_model(model, name="model", skops_trusted_types=trusted)

comparison = pd.DataFrame(results).T.round(3)
comparison

2026/10/23 03:24:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/23 03:24:59 INFO mlflow.store.db.utils: Updating database tables


2026/10/23 03:25:01 INFO mlflow.tracking.fluent: Experiment with name 'abalone-age' does not exist. Creating a new experiment.


2026/10/23 03:25:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/10/23 03:25:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


,rmse,mae,r2
linear_regression,2.212,1.593,0.548
random_forest,2.256,1.585,0.530


## Decision

**Chosen model: `LinearRegression`.**

| model | RMSE | MAE | R2 |
|---|---|---|---|
| LinearRegression | 2.212 | 1.593 | 0.548 |
| RandomForestRegressor (100 trees) | 2.256 | 1.585 | 0.530 |

The random forest is not better: its RMSE is slightly higher (2.256 vs 2.212) and its MAE is almost identical. Since it gives no gain (we would accept the extra complexity only for a gain above ~0.15 RMSE), we keep the simpler, faster and easier to explain linear regression. Both runs are logged in the MLflow experiment `abalone-age`.

To browse the runs, run `uv run mlflow ui --backend-store-uri sqlite:///mlflow.db` from the repo root.
